In [1]:
"""
Agentic Search - POC Notebook Pipeline
=======================================
This script serves as the proof-of-concept, structured like notebook cells.
Each "cell" is a self-contained step that can be run independently.
 
To run: Set ANTHROPIC_API_KEY and TAVILY_API_KEY env vars, then:
    python notebooks/poc_pipeline.py
 
Or import cells individually for interactive exploration.
"""
 
import sys
import os
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)

In [2]:
def cell_1_setup():
    """Initialize configuration and validate API keys."""
    from src.config import Config
 
    config = Config()
 
    print("=== Configuration ===")
    print(f"LLM Model:          {config.llm_model}")
    print(f"Max Search Results:  {config.max_search_results}")
    print(f"Max Entities:        {config.max_entities}")
    print(f"Max Content Chars:   {config.max_content_chars}")
    print(f"Anthropic Key:       {'✓ set' if config.anthropic_api_key else '✗ MISSING'}")
    print(f"Tavily Key:          {'✓ set' if config.tavily_api_key else '✗ MISSING'}")
 
    config.validate()
    print("\n✓ All API keys present. Ready to go!")
    return config

In [3]:
# Cell 1: Setup
config = cell_1_setup()

=== Configuration ===
LLM Model:          claude-haiku-4-5-20251001
Max Search Results:  10
Max Entities:        50
Max Content Chars:   15000
Anthropic Key:       ✓ set
Tavily Key:          ✓ set

✓ All API keys present. Ready to go!


In [4]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 2: Load Query Dataset                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_2_load_queries():
    """Load and explore the query dataset."""
    # data_path = os.path.join(
    #     os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
    #     "data", "queries.json"
    # )
    data_path = os.path.join(
        os.path.abspath(os.path.join(os.getcwd(), "..")),
        "data", "queries.json"
    )
    with open(data_path) as f:
        dataset = json.load(f)
 
    train = dataset["train"]
    eval_set = dataset["eval"]
 
    print(f"=== Query Dataset ===")
    print(f"Train queries:    {len(train)}")
    print(f"Eval queries:     {len(eval_set)}")
 
    # Complexity breakdown
    complexities = {}
    for q in train:
        c = q.get("complexity", "unknown")
        complexities[c] = complexities.get(c, 0) + 1
    print(f"\nTrain complexity: {complexities}")
 
    # Category breakdown
    categories = {}
    for q in train:
        cat = q.get("category", "unknown")
        categories[cat] = categories.get(cat, 0) + 1
    print(f"Train categories: {json.dumps(categories, indent=2)}")
 
    # Show a few examples
    print("\n--- Sample queries ---")
    for q in train[:5]:
        print(f"  [{q['complexity']:8s}] {q['query']}")
        if q.get('post_filters'):
            print(f"             → search: '{q['search_terms']}', filter: {q['post_filters']}")
 
    return dataset

In [5]:
# Cell 2: Load queries
dataset = cell_2_load_queries()

=== Query Dataset ===
Train queries:    40
Eval queries:     15

Train complexity: {'simple': 27, 'complex': 7, 'moderate': 6}
Train categories: {
  "tech_startups": 8,
  "restaurants_food": 6,
  "tools_software": 6,
  "services": 4,
  "education": 4,
  "travel": 3,
  "finance": 2,
  "entertainment": 2,
  "healthcare": 2,
  "shopping": 2,
  "sports": 1
}

--- Sample queries ---
  [simple  ] AI startups in healthcare
  [simple  ] top pizza places in Brooklyn
  [simple  ] open source database tools
  [simple  ] best coworking spaces in Austin
  [simple  ] top machine learning conferences 2025


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 3: Test Query Classification (Stage A)                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_3_test_classification(config):
    """Test the query classifier on sample queries."""
    from src.query_classifier import classify_query
    from src.config import Timer
 
    test_queries = [
        # Should be accepted (topic queries)
        "AI startups in healthcare",
        "best pizza places in Brooklyn",
        "Asian restaurants in Chicago that serve halal food",
        # Should be rejected (not topic queries)
        "What is photosynthesis?",
        "How do I fix a leaky faucet?",
    ]
 
    print("=== Query Classification Tests ===\n")
    for query in test_queries:
        with Timer(f"Classify: {query}"):
            result = classify_query(query, config)
 
        status = "✓ TOPIC" if result.is_topic_query else "✗ REJECT"
        print(f"  [{status}] {query}")
        if result.is_topic_query:
            print(f"    Entity type: {result.entity_type}")
            print(f"    Search:      {result.search_terms}")
            if result.post_filters:
                print(f"    Filters:     {result.post_filters}")
            print(f"    Columns:     {result.suggested_columns}")
        print(f"    Reasoning:   {result.reasoning}")
        print()
 
    return True

In [7]:
# Cell 3: Classification
cell_3_test_classification(config)

=== Query Classification Tests ===



12:33:39 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:33:39 [INFO] agentic_search: [Timer] Classify: AI startups in healthcare: 2.03s


  [✓ TOPIC] AI startups in healthcare
    Entity type: company
    Search:      AI startups in healthcare
    Columns:     ['Company Name', 'Founding Year', 'Focus Area', 'Funding Status', 'Location', 'Key Technology']
    Reasoning:   This is a clear topic query seeking a list of companies. The user wants to discover multiple AI startups operating in the healthcare sector. Both the industry (AI) and vertical (healthcare) are core search parameters, so no post-filters are needed. All constraints are primary category attributes that search engines can reliably filter on.



12:33:40 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:33:40 [INFO] agentic_search: [Timer] Classify: best pizza places in Brooklyn: 1.55s


  [✓ TOPIC] best pizza places in Brooklyn
    Entity type: restaurant
    Search:      best pizza places in Brooklyn
    Columns:     ['name', 'neighborhood', 'cuisine_style', 'price_range', 'rating', 'address', 'phone']
    Reasoning:   This is a clear topic query seeking a list of pizza restaurants in Brooklyn. The location (Brooklyn) and cuisine type (pizza) are primary search criteria that should be in search_terms. No post-filters are needed as the query is straightforward with no uncommon constraints.



...

12:33:44 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:33:44 [INFO] agentic_search: [Timer] Classify: What is photosynthesis?: 1.22s


  [✗ REJECT] What is photosynthesis?
    Reasoning:   This is a factual/definitional question asking for an explanation of a biological process, not a request for a list of entities. It should be answered with a descriptive explanation rather than a structured table.



12:33:46 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:33:46 [INFO] agentic_search: [Timer] Classify: How do I fix a leaky faucet?: 2.06s


  [✗ REJECT] How do I fix a leaky faucet?
    Reasoning:   This is a how-to/instructional question seeking procedural guidance on fixing a specific problem, not a list of entities. It does not ask for a structured set of items (places, products, people, etc.) that could be displayed in a table.



True

In [8]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 4: Test Web Search (Stage B)                                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_4_test_search(config):
    """Test Tavily web search on a sample query."""
    from src.web_searcher import search_web
    from src.config import Timer
 
    query = "AI startups in healthcare"
 
    print(f"=== Web Search Test: '{query}' ===\n")
    with Timer("Web Search"):
        results = search_web(query, config)
 
    print(f"Found {len(results)} results:\n")
    for i, r in enumerate(results):
        print(f"  {i+1}. {r.title}")
        print(f"     URL: {r.url}")
        print(f"     Score: {r.score:.3f}")
        print(f"     Content length: {len(r.content)} chars")
        print(f"     Snippet: {r.snippet[:120]}...")
        print()
 
    return results

In [9]:
# Cell 4: Search
results = cell_4_test_search(config)

=== Web Search Test: 'AI startups in healthcare' ===



12:34:01 [INFO] agentic_search: Search returned 10 results for: AI startups in healthcare
12:34:01 [INFO] agentic_search: [Timer] Web Search: 0.25s


Found 10 results:

  1. Breakthrough AI Startups Making Waves in Healthcare in 2026
     URL: https://www.solutelabs.com/blog/top-ai-healthcare-startups
     Score: 1.000
     Content length: 1941 chars
     Snippet: Despite these challenges, the outlook for healthcare AI remains exceptionally promising. Medical AI companies are expect...

  2. Roadmap: Healthcare AI - Bessemer Venture Partners
     URL: https://www.bvp.com/atlas/roadmap-healthcare-ai
     Score: 1.000
     Content length: 1694 chars
     Snippet: We’ve now reached a critical inflection point in healthcare AI. Today, each new breakthrough in research has the potenti...

  3. 171 Healthcare Startups Funded by Sequoia, YC, A16Z (2026)
     URL: https://topstartups.io/?industries=Healthcare
     Score: 1.000
     Content length: 2380 chars
     Snippet: 51-100 employees Founded: 2021

Funding:

Andreessen Horowitz Y Combinator$30M Series B in 2025

Take action:

See who w...

  4. Healthcare IT Startups funded by Y Combin

In [10]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 5: Test Content Enrichment (Stage C)                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_5_test_enrichment(results, config):
    """Test content enrichment/scraping on search results."""
    from src.content_parser import enrich_content
    from src.config import Timer
 
    print("=== Content Enrichment Test ===\n")
 
    # Show pre-enrichment stats
    for i, r in enumerate(results[:3]):
        print(f"  Before enrichment #{i+1}: {len(r.content)} chars")
 
    with Timer("Content Enrichment"):
        enriched = enrich_content(results, config)
 
    print()
    for i, r in enumerate(enriched[:3]):
        print(f"  After enrichment #{i+1}: {len(r.content)} chars")
        print(f"    Preview: {r.content[:150]}...")
        print()
 
    return enriched

In [11]:
# Cell 5: Enrichment
enriched = cell_5_test_enrichment(results, config)

12:34:07 [INFO] agentic_search: Content enrichment: 10 results, 0 additionally scraped
12:34:07 [INFO] agentic_search: [Timer] Content Enrichment: 0.00s


=== Content Enrichment Test ===

  Before enrichment #1: 1941 chars
  Before enrichment #2: 1694 chars
  Before enrichment #3: 2380 chars

  After enrichment #1: 1941 chars
    Preview: Despite these challenges, the outlook for healthcare AI remains exceptionally promising. Medical AI companies are expected to expand their influence a...

  After enrichment #2: 1694 chars
    Preview: We’ve now reached a critical inflection point in healthcare AI. Today, each new breakthrough in research has the potential to solve real-world problem...

  After enrichment #3: 2380 chars
    Preview: 51-100 employees Founded: 2021

Funding:

Andreessen Horowitz Y Combinator$30M Series B in 2025

Take action:

See who works here 🤝

Check company sit...



In [12]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 6: Test Entity Extraction (Stage C2)                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_6_test_extraction(enriched_results, config):
    """Test LLM-based entity extraction."""
    from src.entity_extractor import extract_entities
    from src.config import QueryAnalysis, Timer
    import pandas as pd
 
    query = "AI startups in healthcare"
    analysis = QueryAnalysis(
        is_topic_query=True,
        entity_type="company",
        search_terms="AI startups in healthcare",
        post_filters=[],
        # suggested_columns=["name", "description", "focus_area", "funding", "location"],
        # suggested_columns=['name', 'description', 'location', 'funding_stage', 'founding_year', 'focus_area', 'website']
        suggested_columns=['Company Name', 'Founding Year', 'Focus Area', 'Funding Status', 'Location', 'Key Technology']
    )
 
    print(f"=== Entity Extraction Test: '{query}' ===\n")
    with Timer("Entity Extraction"):
        entities, columns = extract_entities(query, analysis, enriched_results, config)
 
    print(f"Extracted {len(entities)} entities with columns: {columns}\n")
 
    if entities:
        rows = [e.attributes for e in entities]
        df = pd.DataFrame(rows)
        ordered = [c for c in columns if c in df.columns]
        df = df[ordered]
        print(df.to_string(index=False))
 
        print("\n--- Source tracing (first 3 entities) ---")
        for e in entities[:3]:
            name = e.attributes.get("name", "?")
            sources = set(v for v in e.sources.values() if v)
            print(f"  {name}: {sources}")
 
    return entities, columns

In [13]:
# Cell 6: Extraction
entities, columns = cell_6_test_extraction(enriched, config)

=== Entity Extraction Test: 'AI startups in healthcare' ===



12:35:43 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:35:43 [INFO] agentic_search: Extracted 20 entities with 6 columns
12:35:43 [INFO] agentic_search: [Timer] Entity Extraction: 12.77s


Extracted 20 entities with columns: ['Company Name', 'Founding Year', 'Focus Area', 'Funding Status', 'Location', 'Key Technology']

  Company Name  Founding Year                                     Focus Area         Funding Status                                Location                                                 Key Technology
        Rad AI         2018.0            Radiology report writing automation  $60M Series C in 2025                                  Remote                                    AI automation for radiology
       Qventus         2012.0               Hospital operations optimization  $85M Series D in 2025 San Francisco Bay Area, California, USA                            Predictive analytics for healthcare
        Thatch         2021.0                   Healthcare for startup teams  $38M Series A in 2024                                  Remote                                                  Healthcare AI
  Slingshot AI            NaN                Mental hea

In [14]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 7: Full Pipeline Test                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_7_full_pipeline(config):
    """Run the complete pipeline on several test queries."""
    from src.pipeline import run, display_result
 
    test_queries = [
        "open source database tools",
        "best coffee roasters in Portland",
        # Complex query with post-filter:
        # "vegan restaurants in Los Angeles with outdoor seating",
    ]
 
    print("=" * 70)
    print("FULL PIPELINE TESTS")
    print("=" * 70)
 
    for query in test_queries:
        print(f"\n{'─' * 70}")
        result = run(query, config)
        print(display_result(result))
        print(f"{'─' * 70}\n")
 
    return True

In [15]:
# Cell 7: Full pipeline
cell_7_full_pipeline(config)

FULL PIPELINE TESTS

──────────────────────────────────────────────────────────────────────


12:36:10 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:36:10 [INFO] agentic_search: [Timer] Stage A: Query Classification: 1.83s
12:36:10 [INFO] agentic_search: Query accepted: entity_type=tool, search='open source database tools', post_filters=[]
12:36:10 [INFO] agentic_search: Search returned 10 results for: open source database tools
12:36:10 [INFO] agentic_search: [Timer] Stage B: Web Search: 0.25s
12:36:10 [INFO] agentic_search: Content enrichment: 10 results, 0 additionally scraped
12:36:10 [INFO] agentic_search: [Timer] Stage C: Content Enrichment: 0.00s
12:36:32 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:36:32 [INFO] agentic_search: Extracted 25 entities with 7 columns
12:36:32 [INFO] agentic_search: [Timer] Stage C2: Entity Extraction: 22.48s
12:36:32 [INFO] agentic_search: Pipeline complete: 25 entities, 7 columns, 24.57s


Query: open source database tools
Is topic query: True
Entity type: tool
Search terms: open source database tools
Search results: 10
Entities found: 25
Processing time: 24.57s

            name                                                                                                                                                                    description    primary_language                                                                     license                                                                                               use_cases github_stars maturity_level
      PostgreSQL                                                                    A widely-used open source relational database management system known for reliability and advanced features                 NaN                                                          PostgreSQL License                                                        [OLTP, real-time transactions, web applications]         Non

12:36:34 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:36:34 [INFO] agentic_search: [Timer] Stage A: Query Classification: 1.73s
12:36:34 [INFO] agentic_search: Query accepted: entity_type=company, search='best coffee roasters in Portland', post_filters=[]
12:36:36 [INFO] agentic_search: Search returned 10 results for: best coffee roasters in Portland
12:36:36 [INFO] agentic_search: [Timer] Stage B: Web Search: 1.53s
12:36:36 [INFO] agentic_search: Content enrichment: 10 results, 0 additionally scraped
12:36:36 [INFO] agentic_search: [Timer] Stage C: Content Enrichment: 0.00s
12:36:51 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:36:51 [INFO] agentic_search: Extracted 22 entities with 6 columns
12:36:51 [INFO] agentic_search: [Timer] Stage C2: Entity Extraction: 15.66s
12:36:51 [INFO] agentic_search: Pipeline complete: 22 entities, 6 columns, 18.92s


Query: best coffee roasters in Portland
Is topic query: True
Entity type: company
Search terms: best coffee roasters in Portland
Search results: 10
Entities found: 22
Processing time: 18.92s

                     name                             location                                                    specialty/roast_style  founded_year                                  website rating
         Never Coffee Lab   Hawthorne & Downtown, Portland, OR                        Fruit-forward blends with single-origin offerings           NaN                                      NaN   None
 Sterling Coffee Roasters            NW District, Portland, OR       Single origin from Latin America and Ethiopia, well-rounded blends           NaN             https://www.sterling.coffee/   None
          Marigold Coffee                         Portland, OR                              Single origin coffees from around the world           NaN                                      NaN   None
   Futura Coffee

True

In [16]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 8: Post-Filter Test (Complex Query)                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_8_test_post_filter(config):
    """Test the full pipeline on a complex query with post-filtering."""
    from src.pipeline import run, display_result
 
    query = "Asian restaurants in Chicago that serve halal food"
 
    print(f"=== Complex Query Test: '{query}' ===\n")
    result = run(query, config)
    print(display_result(result))
 
    return result

In [17]:
# Cell 8: Complex query
# Uncomment to test (uses extra API credits):
cell_8_test_post_filter(config)

=== Complex Query Test: 'Asian restaurants in Chicago that serve halal food' ===



12:41:22 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:41:22 [INFO] agentic_search: [Timer] Stage A: Query Classification: 2.16s
12:41:22 [INFO] agentic_search: Query accepted: entity_type=restaurant, search='Asian restaurants in Chicago', post_filters=['serves halal food', 'halal certified or halal-friendly']
12:41:25 [INFO] agentic_search: Search returned 10 results for: Asian restaurants in Chicago
12:41:25 [INFO] agentic_search: [Timer] Stage B: Web Search: 3.11s
12:41:25 [INFO] agentic_search: Scraping: https://www.reddit.com/r/chicagofood/comments/17ig16c/must_eat_asian_food_experienced/
12:41:25 [WARNING] trafilatura.core: discarding data: None
12:41:25 [WARNING] agentic_search: Trafilatura extraction failed for https://www.reddit.com/r/chicagofood/comments/17ig16c/must_eat_asian_food_experienced/
12:41:25 [INFO] agentic_search: Content enrichment: 10 results, 0 additionally scraped
12:41:25 [INFO] agentic_search: [Timer] Stage C: Co

Query: Asian restaurants in Chicago that serve halal food
Is topic query: True
Entity type: restaurant
Search terms: Asian restaurants in Chicago
Post-filters: ['serves halal food', 'halal certified or halal-friendly']
Search results: 10
Entities found: 0
Processing time: 26.22s


PipelineResult(query='Asian restaurants in Chicago that serve halal food', analysis=QueryAnalysis(is_topic_query=True, entity_type='restaurant', search_terms='Asian restaurants in Chicago', post_filters=['serves halal food', 'halal certified or halal-friendly'], suggested_columns=['name', 'cuisine_type', 'address', 'phone', 'website', 'halal_status', 'rating'], reasoning="This is a valid topic query seeking a list of restaurants. Location (Chicago) and primary cuisine (Asian) are core search parameters. However, 'serves halal food' is a specific dietary/certification requirement that search engines won't reliably filter on in results, so it's best applied as a post-filter after retrieving Asian restaurants in Chicago."), entities=[], columns=['name', 'cuisine_type', 'address', 'phone', 'website', 'halal_status', 'rating'], search_results_count=10, processing_time=26.219815254211426, errors=[])

In [18]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 8: Post-Filter Test (Complex Query)                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_8_test_post_filter(config):
    """Test the full pipeline on a complex query with post-filtering."""
    from src.pipeline import run, display_result
 
    query = "Indian veg-only resteraunts in Chicago"
 
    print(f"=== Complex Query Test: '{query}' ===\n")
    result = run(query, config)
    print(display_result(result))
 
    return result

In [19]:
cell_8_test_post_filter(config)

=== Complex Query Test: 'Indian veg-only resteraunts in Chicago' ===



12:43:00 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
12:43:00 [INFO] agentic_search: [Timer] Stage A: Query Classification: 2.10s
12:43:00 [INFO] agentic_search: Query accepted: entity_type=restaurant, search='vegetarian Indian restaurants in Chicago', post_filters=['veg-only menu (no non-vegetarian options)']
12:43:04 [INFO] agentic_search: Search returned 10 results for: vegetarian Indian restaurants in Chicago
12:43:04 [INFO] agentic_search: [Timer] Stage B: Web Search: 4.01s
12:43:04 [INFO] agentic_search: Scraping: https://www.yelp.com/search?find_desc=Vegetarian+Indian+Restaurant&find_loc=Chicago%2C+IL
12:43:04 [WARNING] agentic_search: Scrape failed for https://www.yelp.com/search?find_desc=Vegetarian+Indian+Restaurant&find_loc=Chicago%2C+IL: 403 Client Error: Forbidden for url: https://www.yelp.com/search?find_desc=Vegetarian+Indian+Restaurant&find_loc=Chicago%2C+IL
12:43:04 [INFO] agentic_search: Scraping: https://www.reddit.com/r/c

Query: Indian veg-only resteraunts in Chicago
Is topic query: True
Entity type: restaurant
Search terms: vegetarian Indian restaurants in Chicago
Post-filters: ['veg-only menu (no non-vegetarian options)']
Search results: 10
Entities found: 4
Processing time: 16.53s

                       name        neighborhood                                cuisine type price range  rating                                            website        phone
Annapurna Simply Vegetarian         Rogers Park Indian (South Indian, Gujarati, Vegetarian)        $$$$     7.8                          https://eatannapurna.com/          NaN
                Arya Bhavan               Devon            Indian (Vegan, Gluten-Free, Raw)         NaN     NaN                            https://aryabhavan.com/ 773-274-5800
               Thali Corner                 NaN                         Indian (Vegetarian)         NaN     4.3                                                NaN          NaN
               Simply South 

PipelineResult(query='Indian veg-only resteraunts in Chicago', analysis=QueryAnalysis(is_topic_query=True, entity_type='restaurant', search_terms='vegetarian Indian restaurants in Chicago', post_filters=['veg-only menu (no non-vegetarian options)'], suggested_columns=['name', 'neighborhood', 'cuisine type', 'price range', 'rating', 'website', 'phone'], reasoning="This is a valid topic query seeking a list of restaurants matching specific criteria. 'Indian' and 'vegetarian' are primary attributes that should be in the search terms. The constraint 'veg-only' (strictly vegetarian, not just having vegetarian options) is more nuanced and harder for search engines to filter reliably, so it's marked as a post_filter to be verified by LLM analysis of results."), entities=[Entity(attributes={'name': 'Annapurna Simply Vegetarian', 'neighborhood': 'Rogers Park', 'cuisine type': 'Indian (South Indian, Gujarati, Vegetarian)', 'price range': '$$$$', 'rating': 7.8, 'website': 'https://eatannapurna.co